## Introduction

This notebook serves as the initial data preparation phase for the customer loyalty program analysis project. The Python code within reads two raw CSV files—customer_flight_activity.csv and customer_loyalty_history.csv—and transforms their contents into a format suitable for database ingestion.

The primary objective of this notebook is to generate a set of SQL INSERT statements, which are then used to populate the tables in a PostgreSQL database. The notebook is structured to walk through the entire data pipeline, from raw files to a ready-to-query database:

- Import Libraries: Importing all necessary Python libraries.

- Read and Overview Data: Conducting an initial exploration of the datasets to understand their structure, data types, and potential integrity issues.

- Data Transformation: Processing the raw data to ensure it is correctly formatted and prepared for seamless insertion into the PostgreSQL tables.

This data preparation process lays the foundation for the subsequent SQL-based exploratory data analysis and business insights derived from the project.

In [1]:
import pandas as pd
import psycopg2
from psycopg2 import extras

## Customer Flight Activities Files

In [2]:
# read the data file

def read_data(filepath):
    """read the data files"""
    df = pd.read_csv(filepath)
    return df

filepath = 'Customer_Flight Activity.csv'
cus = read_data(filepath)

cus.head(5)

,Loyalty Number,Year,Month,Flights Booked,Flights with Companions,Total Flights,Distance,Points Accumulated,Points Redeemed,Dollar Cost Points Redeemed
0,100018,2017,1,3,0,3,1521,152.0,0,0
1,100102,2017,1,10,4,14,2030,203.0,0,0
2,100140,2017,1,6,0,6,1200,120.0,0,0
3,100214,2017,1,0,0,0,0,0.0,0,0
4,100272,2017,1,0,0,0,0,0.0,0,0


## Get overview of the data

In [3]:
cus.columns

Index(['Loyalty Number', 'Year', 'Month', 'Flights Booked',
       'Flights with Companions', 'Total Flights', 'Distance',
       'Points Accumulated', 'Points Redeemed', 'Dollar Cost Points Redeemed'],
      dtype='object')

In [4]:
cus.dtypes

Loyalty Number                   int64
Year                             int64
Month                            int64
Flights Booked                   int64
Flights with Companions          int64
Total Flights                    int64
Distance                         int64
Points Accumulated             float64
Points Redeemed                  int64
Dollar Cost Points Redeemed      int64
dtype: object

In [5]:
cus.shape

(405624, 10)

In [6]:
cus.isnull().sum()

Loyalty Number                 0
Year                           0
Month                          0
Flights Booked                 0
Flights with Companions        0
Total Flights                  0
Distance                       0
Points Accumulated             0
Points Redeemed                0
Dollar Cost Points Redeemed    0
dtype: int64

In [7]:
cus.dtypes

Loyalty Number                   int64
Year                             int64
Month                            int64
Flights Booked                   int64
Flights with Companions          int64
Total Flights                    int64
Distance                         int64
Points Accumulated             float64
Points Redeemed                  int64
Dollar Cost Points Redeemed      int64
dtype: object

In [8]:
cus.columns.to_list()

['Loyalty Number',
 'Year',
 'Month',
 'Flights Booked',
 'Flights with Companions',
 'Total Flights',
 'Distance',
 'Points Accumulated',
 'Points Redeemed',
 'Dollar Cost Points Redeemed']

In [9]:
cus.duplicated().sum()

1864

In [10]:
cus["Loyalty Number"].duplicated().sum()

388887

## Tranform the dataset and insert into a PostgreSQL table

In [11]:
def insert_data_into_postgres(filepath, db_config):
    """
    Converts .csv data and directly inserts it into a PostgreSQL table.
    """

    # Define the ORIGINAL column names from the CSV
    original_columns = [
        'Loyalty Number',
        'Year',
        'Month',
        'Flights Booked',
        'Flights with Companions',
        'Total Flights',
        'Distance',
        'Points Accumulated',
        'Points Redeemed',
        'Dollar Cost Points Redeemed'
    ]

    # Define the NEW column names with underscores (for the database)
    new_columns_for_db = [
        'loyalty_number',
        'year',
        'month',
        'flights_booked',
        'flights_with_companions',
        'total_flights',
        'distance',
        'points_accumulated',
        'points_redeemed',
        'dollar_cost_points_redeemed'
    ]

    table_name = "customer_flight_activity"

    conn = None
    cur = None

    try:
        df = pd.read_csv(filepath)

        # Create a dictionary for renaming
        rename_mapping = dict(zip(original_columns, new_columns_for_db))
        df = df.rename(columns=rename_mapping)

        # Convert primary key columns to int *before* dropping duplicates on them
        df['loyalty_number'] = df['loyalty_number'].fillna(-1).astype(int)
        df['year'] = df['year'].fillna(-1).astype(int)
        df['month'] = df['month'].fillna(-1).astype(int)

        # Now, drop duplicates specifically on these primary key columns
        original_rows_count = len(df)
        df = df.drop_duplicates(subset=['loyalty_number', 'year', 'month'])
        rows_after_pk_duplicates = len(df)

        print(f"Original rows in CSV after initial PK type conversion: {original_rows_count}")
        print(f"Rows after dropping duplicates on PK subset: {rows_after_pk_duplicates}")

        # Continue with other type conversions (now that PKs are handled)
        df["flights_booked"] = df["flights_booked"].fillna(-1).astype(int)
        df["flights_with_companions"] = df["flights_with_companions"].fillna(-1).astype(int)
        df["total_flights"] = df["total_flights"].fillna(-1).astype(int)
        df["distance"] = df["distance"].fillna(-1).astype(int)
        df["points_accumulated"] = df["points_accumulated"].fillna(0.0).astype(float)
        df["points_redeemed"] = df["points_redeemed"].fillna(-1).astype(int)
        df["dollar_cost_points_redeemed"] = df["dollar_cost_points_redeemed"].fillna(-1).astype(int)

        # Replace our placeholder with None for NULL in DB for non-PK columns
        for col in [
            'flights_booked', 'flights_with_companions', 'total_flights',
            'distance', 'points_redeemed', 'dollar_cost_points_redeemed'
            ]:
             df[col] = df[col].replace(-1, None)


        # Establish database connection
        conn = psycopg2.connect(**db_config)
        cur = conn.cursor()

        print(f"Truncating table '{table_name}'...")
        cur.execute(f"TRUNCATE TABLE {table_name} RESTART IDENTITY;")
        conn.commit()

        # Prepare column names for the INSERT statement (these are now the new names)
        # We need to explicitly get them from the DataFrame columns, or use new_columns_for_db
        sql_columns = ', '.join(f'"{col}"' for col in new_columns_for_db) # Use the new names directly here

        # Construct the VALUES placeholders: e.g., %s, %s, %s...
        placeholders = ', '.join(['%s'] * len(new_columns_for_db))

        # The full INSERT statement template
        insert_sql = f"INSERT INTO {table_name} ({sql_columns}) VALUES ({placeholders})"

        # Prepare the data as a list of tuples
        # Crucially, here we use the new column names to extract data from the df
        data_to_insert = [tuple(row[col] for col in new_columns_for_db) for index, row in df.iterrows()]

        print(f"Attempting to insert {len(data_to_insert)} rows into '{table_name}'...")

        extras.execute_batch(cur, insert_sql, data_to_insert, page_size=1000)

        conn.commit()
        print(f"Successfully inserted {len(data_to_insert)} rows.")

    except psycopg2.Error as e:
        print(f"Database error: {e}")
        if conn:
            conn.rollback()
    except Exception as e:
        print(f"An unexpected error occurred: {e}")
    finally:
        if cur:
            cur.close()
        if conn:
            conn.close()
            print("Database connection closed.")


# Database Configuration
db_connection_config = {
    "host": "localhost",
    "database": "postgres",
    "user": "postgres",
    "password": "Saipanz11_",
    "port": "5432"
}

# run it
if __name__ == "__main__":
    csv_filepath = 'Customer_Flight Activity.csv'
    insert_data_into_postgres(csv_filepath, db_connection_config)

Original rows in CSV after initial PK type conversion: 405624
Rows after dropping duplicates on PK subset: 401688
Database error: connection to server at "localhost" (::1), port 5432 failed: Connection refused
	Is the server running on that host and accepting TCP/IP connections?
connection to server at "localhost" (127.0.0.1), port 5432 failed: Connection refused
	Is the server running on that host and accepting TCP/IP connections?



## Customer History File

In [12]:
cushis_filepath = 'Customer_Loyalty_History.csv'
cushist = read_data(cushis_filepath)
cushist.sample(5)

,Loyalty Number,Country,Province,City,Postal Code,Gender,Education,Salary,Marital Status,Loyalty Card,CLV,Enrollment Type,Enrollment Year,Enrollment Month,Cancellation Year,Cancellation Month
10682,458864,Canada,New Brunswick,Fredericton,E3B 2H2,Female,Bachelor,68872.0,Married,Star,2480.49,Standard,2018,6,NaN,NaN
2721,262336,Canada,Quebec,Montreal,H4G 3T4,Male,Bachelor,96516.0,Married,Aurora,8693.65,Standard,2013,5,NaN,NaN
12137,812617,Canada,New Brunswick,Fredericton,E3B 2H2,Female,Bachelor,70435.0,Single,Star,3193.46,Standard,2018,9,NaN,NaN
16250,219984,Canada,Quebec,Montreal,H4G 3T4,Female,College,NaN,Divorced,Star,32437.05,2018 Promotion,2018,3,NaN,NaN
4764,235398,Canada,Ontario,Toronto,M2M 7K8,Male,Bachelor,47445.0,Married,Nova,3102.79,Standard,2012,9,NaN,NaN


## Get overview of the data

In [13]:
cushist.describe()

,Loyalty Number,Salary,CLV,Enrollment Year,Enrollment Month,Cancellation Year,Cancellation Month
count,16737.000000,12499.000000,16737.000000,16737.000000,16737.000000,2067.000000,2067.000000
mean,549735.880445,79245.609409,7988.896536,2015.253211,6.669116,2016.503145,6.962748
std,258912.132453,35008.297285,6860.982280,1.979111,3.398958,1.380743,3.455297
min,100018.000000,-58486.000000,1898.010000,2012.000000,1.000000,2013.000000,1.000000
25%,326603.000000,59246.500000,3980.840000,2014.000000,4.000000,2016.000000,4.000000
50%,550434.000000,73455.000000,5780.180000,2015.000000,7.000000,2017.000000,7.000000
75%,772019.000000,88517.500000,8940.580000,2017.000000,10.000000,2018.000000,10.000000
max,999986.000000,407228.000000,83325.380000,2018.000000,12.000000,2018.000000,12.000000


In [14]:
cushist.columns.tolist()

['Loyalty Number',
 'Country',
 'Province',
 'City',
 'Postal Code',
 'Gender',
 'Education',
 'Salary',
 'Marital Status',
 'Loyalty Card',
 'CLV',
 'Enrollment Type',
 'Enrollment Year',
 'Enrollment Month',
 'Cancellation Year',
 'Cancellation Month']

In [15]:
cushist["Cancellation Year"].describe()

count    2067.000000
mean     2016.503145
std         1.380743
min      2013.000000
25%      2016.000000
50%      2017.000000
75%      2018.000000
max      2018.000000
Name: Cancellation Year, dtype: float64

In [16]:
cushist.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16737 entries, 0 to 16736
Data columns (total 16 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Loyalty Number      16737 non-null  int64  
 1   Country             16737 non-null  object 
 2   Province            16737 non-null  object 
 3   City                16737 non-null  object 
 4   Postal Code         16737 non-null  object 
 5   Gender              16737 non-null  object 
 6   Education           16737 non-null  object 
 7   Salary              12499 non-null  float64
 8   Marital Status      16737 non-null  object 
 9   Loyalty Card        16737 non-null  object 
 10  CLV                 16737 non-null  float64
 11  Enrollment Type     16737 non-null  object 
 12  Enrollment Year     16737 non-null  int64  
 13  Enrollment Month    16737 non-null  int64  
 14  Cancellation Year   2067 non-null   float64
 15  Cancellation Month  2067 non-null   float64
dtypes: f

In [17]:
cushist.isnull().sum()

Loyalty Number            0
Country                   0
Province                  0
City                      0
Postal Code               0
Gender                    0
Education                 0
Salary                 4238
Marital Status            0
Loyalty Card              0
CLV                       0
Enrollment Type           0
Enrollment Year           0
Enrollment Month          0
Cancellation Year     14670
Cancellation Month    14670
dtype: int64

In [18]:
cushist.duplicated().sum()

0

In [19]:
cushist.shape

(16737, 16)

In [20]:
def insert_loyalty_history_data(filepath, db_config):
    """
    Converts .csv data for Customer Loyalty History and inserts it into PostgreSQL.
    Handles nulls, type conversions, and bulk insertion.
    """

    # Define the original column names from the CSV
    original_columns = [
        'Loyalty Number', 'Country', 'Province', 'City', 'Postal Code',
        'Gender', 'Education', 'Salary', 'Marital Status', 'Loyalty Card',
        'CLV', 'Enrollment Type', 'Enrollment Year', 'Enrollment Month',
        'Cancellation Year', 'Cancellation Month'
    ]

    # Define new column names to match the database schema
    new_columns_for_db = [
        'loyalty_number', 'country', 'province', 'city', 'postal_code',
        'gender', 'education', 'salary', 'marital_status', 'loyalty_card',
        'clv', 'enrollment_type', 'enrollment_year', 'enrollment_month',
        'cancellation_year', 'cancellation_month'
    ]

    table_name = 'customer_loyalty_history'

    conn = None
    cur = None

    try:
        # Load CSV
        df = pd.read_csv(filepath)

        # Rename columns to match the database
        rename_mapping = dict(zip(original_columns, new_columns_for_db))
        df = df.rename(columns=rename_mapping)

        # Drop duplicates based on primary key
        original_rows_count = len(df)
        df = df.drop_duplicates(subset=['loyalty_number'])
        rows_after_pk_duplicates = len(df)
        print(f"Original rows in CSV: {original_rows_count}")
        print(f"Rows after dropping duplicates on loyalty_number subset: {rows_after_pk_duplicates}")

        # Convert types for consistency with DB schema
        df['loyalty_number'] = df['loyalty_number'].astype('int64')
        df['enrollment_year'] = df['enrollment_year'].astype('int64')
        df['enrollment_month'] = df['enrollment_month'].astype('int64')
        df['cancellation_year'] = df['cancellation_year'].astype('Int64')  # Nullable int
        df['cancellation_month'] = df['cancellation_month'].astype('Int64')  # Nullable int
        df["postal_code"] = df["postal_code"].astype(str)

        # Replace pandas' NA with Python's None
        df = df.replace({pd.NA: None})

        # Connect to database
        conn = psycopg2.connect(**db_config)
        cur = conn.cursor()

        print(f"Truncating table '{table_name}'...")
        cur.execute(f"TRUNCATE TABLE {table_name} RESTART IDENTITY;")
        conn.commit()

        # Prepare SQL insert
        sql_columns = ', '.join(new_columns_for_db)
        placeholders = ', '.join(['%s'] * len(new_columns_for_db))
        insert_sql = f"INSERT INTO {table_name} ({sql_columns}) VALUES ({placeholders})"

        # --- FINAL OPTIMIZATION: Use execute_batch for performance ---
        data_to_insert = [tuple(row[col] for col in new_columns_for_db) for _, row in df.iterrows()]
        
        print(f"Attempting to insert {len(df)} rows one by one...")
        
        for index, row in df.iterrows():
            data_to_insert = tuple(row[col] for col in new_columns_for_db)
            try:
                cur.execute(insert_sql, data_to_insert)
            except psycopg2.Error as e:
                print(f"Error on row index: {index}")
                print(f"Data for this row: {data_to_insert}")
                raise e  # Re-raise the exception to stop the process and see the error
        
        conn.commit()
        print(f"Successfully inserted {len(df)} rows.")

    except psycopg2.Error as e:
        print(f"Database error: {e}")
        if conn:
            conn.rollback()
    except Exception as e:
        print(f"An unexpected error occurred: {e}")
    finally:
        if cur:
            cur.close()
        if conn:
            conn.close()
            print("Database connection closed.")

# Database Configuration
db_connection_config = {
    "host": "localhost",
    "database": "postgres",
    "user": "postgres",
    "password": "Saipanz11_",
    "port": "5432"
}

# Run the import
if __name__ == "__main__":
    csv_filepath = 'Customer_Loyalty_History.csv'
    insert_loyalty_history_data(csv_filepath, db_connection_config)


Original rows in CSV: 16737
Rows after dropping duplicates on loyalty_number subset: 16737
Database error: connection to server at "localhost" (::1), port 5432 failed: Connection refused
	Is the server running on that host and accepting TCP/IP connections?
connection to server at "localhost" (127.0.0.1), port 5432 failed: Connection refused
	Is the server running on that host and accepting TCP/IP connections?



# Here's a quick summary of what we accomplished together:

- Initial Schema Error: You correctly identified that Cancellation Year and Cancellation Month having nulls was expected and not an issue.

- integer out of range (First Time): We correctly diagnosed this as a numeric overflow and resolved it by increasing the precision of your NUMERIC and BIGINT columns.

- integer out of range (Second Time): This was the most challenging part. Your careful Python investigation revealed that the max values were fine, which forced us to look for a subtler issue.

- Debugging applymap: Your sharp eye caught the FutureWarning on applymap, which, combined with the row-by-row debugging, helped us pinpoint the integer out of range error as a side effect of that function.

- can't adapt type 'NAType': This final error message was the last piece of the puzzle. It revealed the specific incompatibility between the pandas Int64 type and the psycopg2 driver, which was fixed by explicitly converting pd.NA to None.


- Now that the data is successfully flowing, you can make one final change to optimize your script's performance. Final Optimization: Switch back to Bulk Insertion.

Now that you've confirmed the data is clean and the insertion logic is sound, you should switch back from the slow, row-by-row loop to the much faster execute_batch method.